# Noetic Adventure — Reproducibility Notebook

**Radin, D., & Cline, R. (2026). Does wavefunction collapse require a conscious observer? A preregistered retrocausal interferometer test in humans and AI agents.**

This notebook reproduces the primary confirmatory analyses and supplementary figures from the paper above, including the regression-residual slope analysis, factorial summary, cumulative Stouffer Z, and lag sensitivity sweep.

**Preregistered protocol (OSF):** `lagSeconds = 5`, `minCorrect = 9`, `totalRequired = 10`, `nPerm = 5000`, gates 1–7.

---

### How to run

**Option A — Google Colab (recommended)**
1. Download the Zenodo archive and unzip it.
2. Upload the entire unzipped folder to your Google Drive.
3. Mount Drive in Colab (see cell below), set `DATA_PATH` to the folder location, and run all cells.

**Option B — Local Jupyter**
1. Unzip the Zenodo archive.
2. Open this notebook from the `colab/` subdirectory.
3. Set `DATA_PATH` in the Setup cell to `".."` (the default) and run all cells.

**Dependencies:** `numpy`, `scipy`, `pandas`, `matplotlib` — all available by default in Colab.

In [ ]:
# Download data files from Zenodo (skipped if already present)
import urllib.request, zipfile, os, shutil
if not os.path.exists('sessions_out.csv'):
    print('Downloading data from Zenodo...')
    urllib.request.urlretrieve(
        'https://zenodo.org/records/20099529/files/NoeticAdventure_colab.zip',
        'data.zip')
    with zipfile.ZipFile('data.zip') as z:
        z.extractall('.')
    os.remove('data.zip')
    # Move files out of colab/ subfolder if needed
    if os.path.isdir('colab') and not os.path.exists('sessions_out.csv'):
        for fname in os.listdir('colab'):
            shutil.move(os.path.join('colab', fname), fname)
        os.rmdir('colab')
    print('Done.')
else:
    print('Data files already present.')

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import sys, os

# Uncomment and edit if running in Google Colab with Drive mounted:
# from google.colab import drive
# drive.mount('/content/drive')
# COLAB_ROOT = '/content/drive/MyDrive/NoeticAdventure'  # adjust to your path
# sys.path.insert(0, os.path.join(COLAB_ROOT, 'colab'))
# DATA_PATH  = COLAB_ROOT

# Default: notebook is inside the colab/ subdirectory, data files are one level up
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
DATA_PATH = '.'

import numpy as np
import matplotlib.pyplot as plt
from load_arrays          import load_arrays
from load_sessions        import load_sessions
from regression_residual  import run_regression_residual
from plot_ensemble        import plot_observed, plot_unobserved
from plot_factorial       import plot_factorial

print('Setup complete.')

---
## 1 · Data

| File | Contents |
|---|---|
| `datatestA.csv` / `datatestB.csv` | z-scored, detrended optical power time series (2 Hz, 1200 samples per segment) |
| `arrays_4000.csv` | Additional segments (4000-series), merged automatically |
| `sessions_out.csv` | Human session metadata — one row per play session |
| `sessions_out_AI.csv` | AI session metadata |

Sessions are linked to sensor segments via the `Section` column (e.g. `a1385` → column `a1385` in sensor A, `b1385` in sensor B).

In [ ]:
# Load sensor arrays (merges datatestA/B.csv + arrays_4000.csv)
A, B = load_arrays(DATA_PATH)
print(f'Sensor arrays: {A.shape[1]} segments × {A.shape[0]} samples each')

# Preview session files
sessions_h,  good_h  = load_sessions(DATA_PATH, source='human',
                                     gates=list(range(1, 8)))
sessions_ai, good_ai = load_sessions(DATA_PATH, source='ai',
                                     gates=list(range(1, 8)))
print(f'\nHuman qualifying sessions (all): {good_h.sum()}')
print(f'AI    qualifying sessions (all): {good_ai.sum()}')

---
## 2 · Human Analysis — Predictions 1 & 2

**Prediction 1:** Humans produce a significant negative slope in the observed (residual) sensor during Intention epochs.

**Prediction 2:** Humans produce a nonsignificant slope difference in the unobserved sensor.

The preregistered stopping rule caps the human cohort at the first **1,000 qualifying sessions** in chronological order.

In [ ]:
res_human = run_regression_residual(
    data_path       = DATA_PATH,
    source          = 'human',
    gates           = list(range(1, 8)),
    max_qualifying  = 1000,   # preregistered human-phase stopping rule
)

In [ ]:
fig1 = plot_observed(res_human, label='Human (N = 1,000)')
plt.show()

fig2 = plot_unobserved(res_human, label='Human (N = 1,000)')
plt.show()

In [ ]:
alpha = 0.05
p1 = res_human['p_block_res']
p2 = res_human['p_block_unobs']
print('── Prediction 1 (Observed, Int-Rel slope diff) ──')
print(f'   p = {p1:.4f}  →  {"SIGNIFICANT" if p1 < alpha else "not significant"}  (predicted: significant)')
print()
print('── Prediction 2 (Unobserved, Int-Rel slope diff) ──')
print(f'   p = {p2:.4f}  →  {"significant (unexpected)" if p2 < alpha else "not significant"}  (predicted: not significant)')

---
## 3 · AI Replication — Prediction 3

**Prediction 3:** AI agents produce nonsignificant slope differences from **both** observed and unobserved sensors.

If predictions 1, 2, and 3 are all confirmed, the result supports the hypothesis that human consciousness — but not current AI — influences quantum interference.

The AI cohort is capped at the first **1,000 qualifying sessions** (see paper for rationale).

In [ ]:
res_ai = run_regression_residual(
    data_path       = DATA_PATH,
    source          = 'ai',
    gates           = list(range(1, 8)),
    max_qualifying  = 1000,
)

In [ ]:
fig3 = plot_observed(res_ai, label='AI (N = 1,000)')
plt.show()

fig4 = plot_unobserved(res_ai, label='AI (N = 1,000)')
plt.show()

In [ ]:
p3_obs   = res_ai['p_block_res']
p3_unobs = res_ai['p_block_unobs']
print('── Prediction 3 (AI — both sensors) ──')
print(f'   Observed   p = {p3_obs:.4f}  →  {"significant (unexpected)" if p3_obs < alpha else "not significant"}  (predicted: not significant)')
print(f'   Unobserved p = {p3_unobs:.4f}  →  {"significant (unexpected)" if p3_unobs < alpha else "not significant"}  (predicted: not significant)')

---
## 4 · 2 × 2 × 2 Factorial Summary

Single-glance view of all eight cells: **agent** (Human / AI) × **sensor** (Observed / Unobserved) × **epoch** (Intention / Relax).

Each cell shows the one-tailed parametric p-value for ensemble slope < 0, from OLS regression on the ensemble mean curve, using **raw (unregressed) sensor values**. Showing raw values here — rather than regression residuals — confirms that the unobserved sensor is genuinely flat: any signal in the observed sensor cannot be attributed to a shared artifact that would also appear in the unobserved channel. The preregistered confirmatory test (Section 2) uses the regression-residual method to formally remove shared laser variance.

In [ ]:
fig5 = plot_factorial(res_human, res_ai)
plt.show()

---
## 5 · Overall Verdict

In [ ]:
pred1 = res_human['p_block_res']   < alpha
pred2 = res_human['p_block_unobs'] >= alpha
pred3 = (res_ai['p_block_res'] >= alpha) and (res_ai['p_block_unobs'] >= alpha)

print('═' * 55)
print('PREREGISTERED PREDICTIONS — FINAL VERDICT')
print('═' * 55)
print(f'  Prediction 1 (Human Observed significant):     {"CONFIRMED" if pred1 else "not confirmed"}')
print(f'  Prediction 2 (Human Unobserved null):          {"CONFIRMED" if pred2 else "not confirmed"}')
print(f'  Prediction 3 (AI both sensors null):           {"CONFIRMED" if pred3 else "not confirmed"}')
print('─' * 55)
all_confirmed = pred1 and pred2 and pred3
print(f'  All three predictions confirmed: {all_confirmed}')
print('═' * 55)

print(f'\nKey statistics (Human, N = 1,000 sessions):')
print(f'  Residual Int-Rel slope diff:  {res_human["slope_ri"].mean() - res_human["slope_rr"].mean():.6f}')
print(f'  Cohen\'s d:                    {res_human["cohens_d_res"]:.4f}')
print(f'  Permutation p (label-swap):   {res_human["p_block_res"]:.4f}')
print(f'  Permutation p (ensemble):     {res_human["p_ens_res"]:.4f}')
print(f'\nKey statistics (AI, N = 1,000 sessions):')
print(f'  Residual Int-Rel slope diff:  {res_ai["slope_ri"].mean() - res_ai["slope_rr"].mean():.6f}')
print(f'  Cohen\'s d:                    {res_ai["cohens_d_res"]:.4f}')
print(f'  Permutation p (label-swap):   {res_ai["p_block_res"]:.4f}')

---
## S1 · Cumulative Stouffer Z and Odds Against Chance

Each qualifying session yields ten (Intention - Relax) slope differences.
An **exact sign-flip permutation** over all 2¹⁰ = 1,024 sign patterns
converts each session to a one-tailed z-score (continuity-corrected).

**Cumulative Stouffer Z** = cumsum(z) / √n accumulates evidence over the
chronological session sequence. The shaded band shows the 90% range of
where the curve would sit under 200 random session orderings.

**Odds against chance** = 1/p where p = 1 - Φ(Z), plotted on a log scale
so the contrast between Human and AI is immediately visible.

In [ ]:
from plot_cumulative_z import compute_session_z
from plot_odds         import plot_odds
from plot_cumulative_z import plot_cumulative_z

print("Computing per-session z-scores (Human)...")
zO_H, zU_H = compute_session_z(DATA_PATH, "human", list(range(1, 8)), 1000)
print(f"  {len(zO_H)} valid sessions")

print("\nComputing per-session z-scores (AI)...")
zO_AI, zU_AI = compute_session_z(DATA_PATH, "ai", list(range(1, 8)), 1000)
print(f"  {len(zO_AI)} valid sessions")

In [ ]:
fig_s1a = plot_cumulative_z(zO_H, zU_H, zO_AI, zU_AI,
                            save_path="cumulative_stouffer_z.png")
plt.show()
plt.close("all")

In [ ]:
fig_s1b = plot_odds(zO_H, zU_H, zO_AI, zU_AI,
                    save_path="odds_against_chance.png")
plt.show()
plt.close("all")

---
## S2 · Lag Sensitivity Analysis

The preregistered cognitive-switching delay is **5 seconds**.
This panel sweeps the delay from 0 to 10 s in 0.5-second steps and plots
odds against chance (one-tailed label-swap permutation, 5,000 iterations)
for both sensors. The vertical dashed line marks the preregistered value.

> **Note:** this cell takes ~10 minutes to run (21 lags × 5,000 permutations).

In [ ]:
from plot_lag_sweep import plot_lag_sweep

fig_s2, lag_results = plot_lag_sweep(
    DATA_PATH, source="human",
    gates=list(range(1, 8)), max_qualifying=1000,
    n_perm=5000,
    save_path="lag_sweep.png",
)
plt.show()
plt.close("all")